# Libraries and Mounting Collab

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from PIL import Image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.applications import ResNet50, EfficientNetB3
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2
import kagglehub

# Image Classification Model

## Loading Raw Data (Skin Problems Dataset)

In [ ]:
base_path = '/content/drive/MyDrive/skin_problems_dataset'
train_dir = os.path.join(base_path, 'train')
valid_dir = os.path.join(base_path, 'valid')
test_dir  = os.path.join(base_path, 'test')

train_df = pd.read_csv(os.path.join(train_dir, '_classes.csv'))
valid_df = pd.read_csv(os.path.join(valid_dir, '_classes.csv'))
test_df  = pd.read_csv(os.path.join(test_dir, '_classes.csv'))

train_df['filename'] = train_df['filename'].apply(lambda x: os.path.join(train_dir, x))
valid_df['filename'] = valid_df['filename'].apply(lambda x: os.path.join(valid_dir, x))
test_df['filename']  = test_df['filename'].apply(lambda x: os.path.join(test_dir, x))

## Exploratory Analysis

In [ ]:
print("Train shape:", train_df.shape)
print("Validation shape:", valid_df.shape)
print("Test shape:", test_df.shape)
print("\nMissing values in train set:")
print(train_df.isnull().sum())
train_df.head()

In [ ]:
label_counts = train_df.drop(columns=['filename']).sum().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
label_counts.plot(kind='bar')
plt.title("📊 Class Distribution in Training Set")
plt.ylabel("Number of Images")
plt.xticks(rotation=45)
plt.grid(axis='y')
plt.tight_layout()
plt.show()

In [ ]:
co_matrix = train_df.drop(columns=['filename']).T.dot(train_df.drop(columns=['filename']))
plt.figure(figsize=(10, 8))
sns.heatmap(co_matrix, annot=True, fmt="d", cmap="Blues")
plt.title("Label Co-Occurrence Matrix")
plt.show()

In [ ]:
def plot_samples_per_class(df, class_columns, num_samples=3):
    for cls in class_columns:
        cls_df = df[df[cls] == 1]
        sample_paths = cls_df['filename'].sample(min(num_samples, len(cls_df)), random_state=42)

        plt.figure(figsize=(12, 4))
        for i, path in enumerate(sample_paths):
            img = Image.open(path)
            plt.subplot(1, num_samples, i+1)
            plt.imshow(img)
            plt.axis('off')
            plt.title(cls)
        plt.suptitle(f"Samples for: {cls}")
        plt.show()

class_columns = [col for col in train_df.columns if col != 'filename']

plot_samples_per_class(train_df, class_columns, num_samples=3)

In [ ]:
train_df['num_labels'] = train_df[class_columns].sum(axis=1)

plt.figure(figsize=(6, 4))
train_df['num_labels'].value_counts().sort_index().plot(kind='bar')
plt.title("Number of Labels per Image")
plt.xlabel("Labels per Image")
plt.ylabel("Number of Images")
plt.grid(axis='y')
plt.tight_layout()
plt.show()

## Data Cleaning

In [ ]:
train_df.columns = train_df.columns.str.strip()
valid_df.columns = valid_df.columns.str.strip()
test_df.columns  = test_df.columns.str.strip()

def drop_blackheads_redness(df):
    return df[(df['Blackheads'] == 0) & (df['Skin Redness'] == 0)]

train_df = drop_blackheads_redness(train_df)
valid_df = drop_blackheads_redness(valid_df)
test_df  = drop_blackheads_redness(test_df)

In [ ]:
class_columns = ['Acne', 'Dark Spots', 'Dry Skin', 'Eye bags',
                 'Normal Skin', 'Oily Skin', 'Pores', 'Wrinkles']
zero_label_rows = train_df[class_columns].sum(axis=1) == 0
print("Images with no labels:", zero_label_rows.sum())

In [ ]:
missing_files = [f for f in train_df['filename'] if not os.path.exists(f)]
print("Missing image files:", len(missing_files))

In [ ]:
train_df['num_labels'] = train_df[class_columns].sum(axis=1)

plt.figure(figsize=(6, 4))
train_df['num_labels'].value_counts().sort_index().plot(kind='bar')
plt.title("Number of Labels per Image")
plt.xlabel("Labels per Image")
plt.ylabel("Number of Images")
plt.grid(axis='y')
plt.tight_layout()
plt.show()

label_counts = train_df[class_columns].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
label_counts.plot(kind='bar')
plt.title("Class Distribution in Training Set")
plt.ylabel("Number of Images")
plt.xticks(rotation=45)
plt.grid(axis='y')
plt.tight_layout()
plt.show()

In [ ]:
print("Train:")
print(train_df_clean[['Blackheads', 'Skin Redness']].sum())
print("\nValidation:")
print(valid_df_clean[['Blackheads', 'Skin Redness']].sum())
print("\nTest:")
print(test_df_clean[['Blackheads', 'Skin Redness']].sum())

## Training the CNN Models

### Set-Up

In [ ]:
class_columns = ['Acne', 'Dark Spots', 'Dry Skin', 'Eye bags',
                 'Normal Skin', 'Oily Skin', 'Pores', 'Wrinkles']

model_configs = {
    "resnet50": {"img_size": (224, 224)},
    "efficientnetb3": {"img_size": (300, 300)},
    "mobilenetv2": {"img_size": (224, 224)}
}

In [ ]:
class_totals = train_df[class_columns].sum()
max_count = max(class_totals)
class_weights = {i: max_count / class_totals[i] for i in range(len(class_columns))}

print("Class Weights (inverse frequency):")
for i, w in class_weights.items():
    print(f"{class_columns[i]}: {w:.2f}")

In [ ]:
def get_weighted_binary_crossentropy(class_weights):
    def loss_fn(y_true, y_pred):
        loss = 0
        for i in range(len(class_columns)):
            weight = class_weights[i]
            loss += -(
                weight * y_true[:, i] * K.log(K.clip(y_pred[:, i], 1e-7, 1)) +
                (1 - y_true[:, i]) * K.log(K.clip(1 - y_pred[:, i], 1e-7, 1))
            )
        return K.mean(loss, axis=-1)
    return loss_fn

In [ ]:
def macro_f1(y_true, y_pred):
    y_true = K.cast(y_true, "float32")
    y_pred = K.cast(y_pred > 0.5, "float32")

    tp = K.sum(y_true * y_pred, axis=0)
    fp = K.sum((1 - y_true) * y_pred, axis=0)
    fn = K.sum(y_true * (1 - y_pred), axis=0)

    precision = tp / (tp + fp + K.epsilon())
    recall = tp / (tp + fn + K.epsilon())
    f1 = 2 * precision * recall / (precision + recall + K.epsilon())

    return K.mean(f1)

### Pre-training

In [ ]:
def build_model(model_name, num_classes, input_shape):
    base_model = None
    if model_name == "resnet50":
        base_model = ResNet50(include_top=False, weights="imagenet", input_shape=input_shape)
    elif model_name == "efficientnetb3":
        base_model = EfficientNetB3(include_top=False, weights="imagenet", input_shape=input_shape)
    elif model_name == "mobilenetv2":
        base_model = MobileNetV2(include_top=False, weights="imagenet", input_shape=input_shape)

    base_model.trainable = False

    x = GlobalAveragePooling2D()(base_model.output)
    x = Dropout(0.4)(x)
    output = Dense(num_classes, activation="sigmoid")(x)

    model = Model(inputs=base_model.input, outputs=output)

    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.AUC(name="auc"), macro_f1]
    )
    return model

In [ ]:
def get_callbacks(model_name):
    early_stop = EarlyStopping(
        monitor="val_auc", patience=5, restore_best_weights=True, mode="max"
    )

    checkpoint_cb = ModelCheckpoint(
        filepath=f"{model_name}_best_pretrain.keras",
        save_best_only=True,
        monitor="val_auc",
        mode="max"
    )

    return [early_stop, checkpoint_cb]

In [ ]:
def train_with_model(model_name):
    print(f"🔹 Training {model_name.upper()}...\n")

    img_size = model_configs[model_name]["img_size"]
    input_shape = img_size + (3,)

    datagen = ImageDataGenerator(rescale=1./255)

    def build_generator(df):
        return datagen.flow_from_dataframe(
            df,
            x_col="filename",
            y_col=class_columns,
            target_size=img_size,
            batch_size=16,
            class_mode="raw",
            shuffle=True
        )

    train_gen = build_generator(train_df)
    val_gen = build_generator(valid_df)

    model = build_model(model_name, num_classes=len(class_columns), input_shape=input_shape)

    weighted_loss = get_weighted_binary_crossentropy(class_weights)
    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss=weighted_loss,
        metrics=[macro_f1, tf.keras.metrics.AUC(name='auc')]
    )

    callbacks = get_callbacks(model_name)

    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=3,
        steps_per_epoch=50,
        validation_steps=val_gen.samples // val_gen.batch_size,
        callbacks=callbacks
    )

    model.save(f"{model_name}1_quick.keras")
    return model, history

In [ ]:
class_columns = ['Acne', 'Dark Spots', 'Dry Skin', 'Eye bags',
                 'Normal Skin', 'Oily Skin', 'Pores', 'Wrinkles']
train_df['num_labels'] = train_df[class_columns].sum(axis=1)

In [ ]:
for model_name in ["mobilenetv2"]:
    model, history = train_with_model(model_name)

In [ ]:
from sklearn.metrics import classification_report, f1_score, roc_auc_score
import numpy as np

def evaluate_model(model, model_name):
    print(f"\nEvaluating {model_name.upper()} on test set...")

    img_size = model_configs[model_name]["img_size"]
    test_gen = ImageDataGenerator(rescale=1./255).flow_from_dataframe(
        test_df,
        x_col="filename",
        y_col=class_columns,
        target_size=img_size,
        batch_size=16,
        class_mode="raw",
        shuffle=False
    )

    y_true = test_gen.labels
    y_pred_prob = model.predict(test_gen)
    y_pred = (y_pred_prob > 0.5).astype(int)

    print("y_true shape:", y_true.shape)
    print("y_pred shape:", y_pred.shape)

    support_mask = y_true.sum(axis=0) > 0
    filtered_class_columns = [class_columns[i] for i, keep in enumerate(support_mask) if keep]

    y_true_filtered = y_true[:, support_mask]
    y_pred_filtered = y_pred[:, support_mask]
    y_pred_prob_filtered = y_pred_prob[:, support_mask]

    print(f"\nEvaluating only {len(filtered_class_columns)} classes with label support:")
    print(filtered_class_columns)

    print("\nClassification Report:")
    print(classification_report(
        y_true_filtered,
        y_pred_filtered,
        target_names=filtered_class_columns,
        zero_division=0
    ))

    # F1 score
    macro_f1 = f1_score(y_true_filtered, y_pred_filtered, average='macro')
    print(f"\nMacro F1 Score: {macro_f1:.4f}")

    # AUC
    try:
        auc = roc_auc_score(y_true_filtered, y_pred_prob_filtered, average='macro')
        print(f"AUC (macro, valid classes): {auc:.4f}")
    except ValueError as e:
        print("AUC computation failed:", e)

In [ ]:
for model_name in ["mobilenetv2"]:
    model = tf.keras.models.load_model(
        f"{model_name}_best_pretrain.keras",
        custom_objects={"macro_f1": macro_f1},
        compile=False
    )
    model.compile(optimizer=Adam(1e-4),
                  loss="binary_crossentropy",
                  metrics=[macro_f1, tf.keras.metrics.AUC(name="auc")])

    evaluate_model(model, model_name)

### Fine-Tuning Best Model (MobileNetV2)

In [ ]:
from tensorflow.keras.metrics import AUC

img_size = (224, 224)
BATCH_SIZE = 16

datagen = ImageDataGenerator(rescale=1.0 / 255)

def build_generator(df):
    return datagen.flow_from_dataframe(
        df,
        x_col="filename",
        y_col=class_columns,
        target_size=img_size,
        batch_size=BATCH_SIZE,
        class_mode="raw",
        shuffle=True
    )

train_gen = build_generator(train_df)
val_gen = build_generator(valid_df)

model = load_model(
    "mobilenetv2_best_pretrain1.keras",
    custom_objects={"macro_f1": macro_f1},
    compile=False
)

unfreeze_n = 30
for layer in model.layers[-unfreeze_n:]:
    layer.trainable = True

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=[macro_f1, AUC(name="auc")]
)

early_stop = EarlyStopping(monitor="val_macro_f1", patience=5, restore_best_weights=True, mode="max")
checkpoint_cb = ModelCheckpoint(
    "mobilenetv2_finetuned_best.keras",
    monitor="val_macro_f1",
    save_best_only=True,
    mode="max"
)

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    steps_per_epoch=len(train_gen),
    validation_steps=len(val_gen),
    callbacks=[early_stop, checkpoint_cb]
)

In [ ]:
import os
drive_path = "/content/drive/MyDrive/skin_model_backup"
os.makedirs(drive_path, exist_ok=True)

from tensorflow.keras.models import load_model
from tensorflow.keras import backend as K

def macro_f1(y_true, y_pred):
    y_pred = K.cast(y_pred > 0.5, "float32")
    tp = K.sum(y_true * y_pred, axis=0)
    fp = K.sum((1 - y_true) * y_pred, axis=0)
    fn = K.sum(y_true * (1 - y_pred), axis=0)
    precision = tp / (tp + fp + K.epsilon())
    recall = tp / (tp + fn + K.epsilon())
    f1 = 2 * precision * recall / (precision + recall + K.epsilon())
    return K.mean(f1)

model = load_model("mobilenetv2_finetuned_best1.keras", custom_objects={"macro_f1": macro_f1})
model.save(f"{drive_path}/mobilenetv2_finetuned_best1.keras")

print("Model saved to Google Drive!")

In [ ]:
import matplotlib.pyplot as plt

def plot_training_history(history):
    metrics = ["loss", "macro_f1", "auc"]

    for metric in metrics:
        plt.figure(figsize=(6, 4))
        plt.plot(history.history[metric], label=f"Train {metric}")
        plt.plot(history.history[f"val_{metric}"], label=f"Val {metric}")
        plt.xlabel("Epochs")
        plt.ylabel(metric.capitalize())
        plt.title(f"{metric.capitalize()} Over Epochs")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()

plot_training_history(history)

Comprehensive Evaluation on Test Set

In [ ]:
def macro_f1(y_true, y_pred):
    y_true = K.cast(y_true, "float32")
    y_pred = K.cast(y_pred > 0.5, "float32")

    tp = K.sum(y_true * y_pred, axis=0)
    fp = K.sum((1 - y_true) * y_pred, axis=0)
    fn = K.sum(y_true * (1 - y_pred), axis=0)

    precision = tp / (tp + fp + K.epsilon())
    recall = tp / (tp + fn + K.epsilon())
    f1 = 2 * precision * recall / (precision + recall + K.epsilon())

    return K.mean(f1)

In [ ]:
from sklearn.metrics import f1_score, roc_auc_score, classification_report

def evaluate_model(model, model_name="Model"):
    print(f"\nEvaluating {model_name.upper()} on test set...")

    test_gen = datagen.flow_from_dataframe(
        test_df,
        x_col="filename",
        y_col=class_columns,
        target_size=img_size,
        batch_size=BATCH_SIZE,
        class_mode="raw",
        shuffle=False
    )

    y_true = test_df[class_columns].values

    y_pred = model.predict(test_gen)
    y_pred_bin = (y_pred > 0.5).astype(int)

    macro_f1_val = f1_score(y_true, y_pred_bin, average="macro", zero_division=0)
    auc_val = roc_auc_score(y_true, y_pred, average="macro")

    print(f"Macro F1: {macro_f1_val:.4f}")
    print(f"AUC: {auc_val:.4f}")

In [ ]:
model = load_model("mobilenetv2_finetuned_best1.keras", custom_objects={"macro_f1": macro_f1})
evaluate_model(model, model_name="mobilenetv2_finetuned1")

In [ ]:
from sklearn.metrics import classification_report

from tensorflow.keras.preprocessing.image import ImageDataGenerator

img_size = (224, 224)

test_datagen = ImageDataGenerator(rescale=1./255)
test_gen = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col="filename",
    y_col=class_columns,
    target_size=img_size,
    batch_size=16,
    class_mode="raw",
    shuffle=False
)

y_true = test_gen.labels
y_pred_prob = model.predict(test_gen)
y_pred = (y_pred_prob > 0.5).astype(int)


report = classification_report(y_true, y_pred, target_names=class_columns, zero_division=0, output_dict=True)
report_df = pd.DataFrame(report).transpose().sort_values("f1-score", ascending=False)
report_df[["precision", "recall", "f1-score"]]

In [ ]:
class_counts = train_df[class_columns].sum().sort_values(ascending=False)

import matplotlib.pyplot as plt

print("Image count per class in training set:\n")
display(class_counts)

plt.figure(figsize=(10, 5))
class_counts.plot(kind='bar', color='skyblue')
plt.title("Number of Images per Class (Train Set)")
plt.ylabel("Image Count")
plt.xticks(rotation=45)
plt.grid(axis="y")
plt.tight_layout()
plt.show()

Since they are underrepesented, we will do data augmentation in an attempt to increase the very low F1 scores for this classes. The plan is to have a around 500 images per class.

### Fine tuning again with Class-Targeted Augmentation

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.metrics import AUC
import pandas as pd

def augment_class(df, class_name, factor=3):
    class_df = df[df[class_name] == 1]
    augmented = pd.concat([class_df] * (factor - 1), ignore_index=True)
    return pd.concat([df, augmented], ignore_index=True)

train_df_augmented = train_df.copy()

weak_classes = ['Eye bags', 'Oily Skin', 'Pores']

for cls in weak_classes:
    train_df_augmented = augment_class(train_df_augmented, cls, factor=3)

print(f"Augmented training size: {len(train_df_augmented)}")

aug_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=25,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    shear_range=0.1,
    brightness_range=(0.8, 1.2),
    horizontal_flip=True
)
val_datagen = ImageDataGenerator(rescale=1.0 / 255)

def build_generator(df, augment=True):
    datagen = aug_datagen if augment else val_datagen
    return datagen.flow_from_dataframe(
        df,
        x_col="filename",
        y_col=class_columns,
        target_size=(224, 224),
        batch_size=16,
        class_mode="raw",
        shuffle=True
    )

train_gen = build_generator(train_df_augmented, augment=True)
val_gen = build_generator(valid_df, augment=False)

model = load_model(
    "mobilenetv2_best_pretrain.keras",
    custom_objects={"macro_f1": macro_f1},
    compile=False
)

unfreeze_n = 30
for layer in model.layers[-unfreeze_n:]:
    layer.trainable = True

weighted_loss = get_weighted_binary_crossentropy(class_weights)
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss=weighted_loss,
    metrics=[macro_f1, AUC(name="auc")]
)

early_stop = EarlyStopping(monitor="val_macro_f1", patience=5, mode="max", restore_best_weights=True)
checkpoint = ModelCheckpoint(
    "mobilenetv2_finetuned_best1.keras",
    monitor="val_macro_f1",
    mode="max",
    save_best_only=True
)

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    steps_per_epoch=len(train_gen),
    validation_steps=len(val_gen),
    callbacks=[early_stop, checkpoint]
)

In [ ]:
label_counts_aug = train_df_augmented[class_columns].sum().sort_values(ascending=False)
print("\nClass counts after augmentation:")
print(label_counts_aug)

In [ ]:
evaluate_model(model, model_name="mobilenetv2_finetuned_aguementation")

In [ ]:
from sklearn.metrics import classification_report
from tensorflow.keras.preprocessing.image import ImageDataGenerator

img_size = (224, 224)
test_datagen = ImageDataGenerator(rescale=1./255)
test_gen = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col="filename",
    y_col=class_columns,
    target_size=img_size,
    batch_size=16,
    class_mode="raw",
    shuffle=False
)

y_true = test_gen.labels
y_pred_prob = model.predict(test_gen)
y_pred = (y_pred_prob > 0.5).astype(int)

report = classification_report(y_true, y_pred, target_names=class_columns, zero_division=0, output_dict=True)
report_df = pd.DataFrame(report).transpose().sort_values("f1-score", ascending=False)
report_df[["precision", "recall", "f1-score"]]

In [ ]:
import os
drive_path = "/content/drive/MyDrive/skin_model_backup"
os.makedirs(drive_path, exist_ok=True)

from tensorflow.keras.models import load_model
from tensorflow.keras import backend as K

def macro_f1(y_true, y_pred):
    y_pred = K.cast(y_pred > 0.5, "float32")
    tp = K.sum(y_true * y_pred, axis=0)
    fp = K.sum((1 - y_true) * y_pred, axis=0)
    fn = K.sum(y_true * (1 - y_pred), axis=0)
    precision = tp / (tp + fp + K.epsilon())
    recall = tp / (tp + fn + K.epsilon())
    f1 = 2 * precision * recall / (precision + recall + K.epsilon())
    return K.mean(f1)

model.save(f"{drive_path}/mobilenetv2_finetuned_augmentation.keras")

print("Model saved to Google Drive!")

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, classification_report

print("Predicting on validation set...")
y_val_true = val_gen.labels
y_val_pred_prob = model.predict(val_gen)

def tune_thresholds(y_true, y_pred_prob, class_names, step=0.01):
    thresholds = {}
    for i, class_name in enumerate(class_names):
        best_f1 = 0
        best_thresh = 0.5
        for t in np.arange(0.1, 0.91, step):
            y_pred_bin = (y_pred_prob[:, i] > t).astype(int)
            f1 = f1_score(y_true[:, i], y_pred_bin, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_thresh = t
        thresholds[class_name] = best_thresh
        print(f"{class_name}: Best F1 = {best_f1:.4f} at threshold = {best_thresh:.2f}")
    return thresholds

print("Tuning thresholds...")
optimal_thresholds = tune_thresholds(y_val_true, y_val_pred_prob, class_columns)

# 3. Predict on test set
print("\n🔍 Predicting on test set...")
y_test_true = test_df[class_columns].values
y_test_pred_prob = model.predict(test_gen)

# 4. Apply optimized thresholds
print("Applying optimized thresholds...")
y_test_pred_bin = np.zeros_like(y_test_pred_prob)
for i, class_name in enumerate(class_columns):
    thresh = optimal_thresholds.get(class_name, 0.5)
    y_test_pred_bin[:, i] = (y_test_pred_prob[:, i] > thresh).astype(int)

# 5. Final classification report
print("\nFinal Evaluation with Tuned Thresholds:")
print(classification_report(
    y_test_true,
    y_test_pred_bin,
    target_names=class_columns,
    zero_division=0
))

In [ ]:
manual_thresholds = {
    'Acne': 0.50,
    'Normal Skin': 0.50,
    'Dry Skin': 0.40,
    'Eye bags': 0.35,
    'Oily Skin': 0.45,
    'Pores': 0.40,
    'Dark Spots': 0.40,
    'Wrinkles': 0.40,
}

In [ ]:
# Re-apply predictions using manual thresholds
y_test_pred_bin_manual = np.zeros_like(y_test_pred_prob)
for i, class_name in enumerate(class_columns):
    thresh = manual_thresholds.get(class_name, 0.5)
    y_test_pred_bin_manual[:, i] = (y_test_pred_prob[:, i] > thresh).astype(int)

# Evaluate
print("\nManual Threshold Evaluation:")
from sklearn.metrics import classification_report
print(classification_report(
    y_test_true,
    y_test_pred_bin_manual,
    target_names=class_columns,
    zero_division=0
))

In [ ]:
# In training notebook
model.save("/content/drive/MyDrive/DermaVue Project/mobilenetv2_final_manual_thresh.keras")

# Recommendation System

Dataset: https://www.kaggle.com/code/eward96/skincare-recommendation-engine

## Loading Raw Data (Skin Products and their Ingredients Dataset)

In [ ]:
path = kagglehub.dataset_download("eward96/skincare-products-clean-dataset")
print("Path to dataset files:", path)

In [ ]:
import shutil
import os

original_path = "/root/.cache/kagglehub/datasets/eward96/skincare-products-clean-dataset/versions/1"

files = os.listdir(original_path)
for f in files:
    if f.endswith(".csv"):
        src = os.path.join(original_path, f)
        dst = f"/content/{f}"
        shutil.copy(src, dst)
        print(f"Copied: {f} to /content/")


In [ ]:
import pandas as pd

df_products = pd.read_csv("/content/skincare_products_clean.csv")
df_products.head()

## Exploratory Analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import ast

print("Dataset Shape:", df_products.shape)
print("\nColumn Names:", df_products.columns.tolist())
print("\nMissing Values:\n", df_products.isnull().sum())

In [ ]:
df_products["price_clean"] = df_products["price"].str.replace("£", "").astype(float)
df_products["clean_ingreds"] = df_products["clean_ingreds"].apply(lambda x: ast.literal_eval(str(x)))
df_products.dropna(subset=["product_name", "clean_ingreds", "price_clean"], inplace=True)
print("Data cleaned!")

In [ ]:
print("\nProduct Types Breakdown:\n", df_products['product_type'].value_counts())

plt.figure(figsize=(12, 5))
sns.countplot(data=df_products, y="product_type", order=df_products["product_type"].value_counts().index)
plt.title("Distribution of Product Types")
plt.xlabel("Count")
plt.ylabel("Product Type")
plt.grid(True)
plt.show()

print("\nPrice Summary:\n", df_products["price_clean"].describe())

plt.figure(figsize=(10, 5))
sns.histplot(df_products["price_clean"], bins=30, kde=True, color="green")
plt.title("Price Distribution")
plt.xlabel("Price (€)")
plt.grid(True)
plt.show()

plt.figure(figsize=(14, 6))
sns.boxplot(data=df_products, x="product_type", y="price_clean")
plt.title("Price by Product Type")
plt.xticks(rotation=45)
plt.grid(True)
plt.show()

# Ingredient Analysis
df_products["num_ingredients"] = df_products["clean_ingreds"].apply(len)

print("\nIngredient Count Stats:\n", df_products["num_ingredients"].describe())

plt.figure(figsize=(10, 4))
sns.histplot(df_products["num_ingredients"], bins=30, kde=True)
plt.title("Number of Ingredients per Product")
plt.xlabel("Ingredient Count")
plt.grid(True)
plt.show()

all_ingredients = [ing for sublist in df_products["clean_ingreds"] for ing in sublist]
ingredient_counts = Counter(all_ingredients).most_common(20)
common_df = pd.DataFrame(ingredient_counts, columns=["Ingredient", "Count"])

plt.figure(figsize=(12, 6))
sns.barplot(data=common_df, x="Count", y="Ingredient", palette="viridis")
plt.title("Top 20 Most Common Ingredients")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
sns.scatterplot(data=df_products, x="num_ingredients", y="price_clean")
plt.title("Ingredient Count vs Price")
plt.xlabel("Number of Ingredients")
plt.ylabel("Price (£)")
plt.grid(True)
plt.show()